In [ ]:
!pip install pandas numpy openai sentence-transformers datasets transformers[torch] accelerate>=1.1.0 seaborn hdbscan umap-learn nltk plotly optuna BERTopic "git+https://github.com/FelSiq/DBCV"

In [28]:
%load_ext cuml.accel

cuML: Installed accelerator for sklearn.
cuML: Installed accelerator for umap.
cuML: Installed accelerator for hdbscan.
cuML: Successfully initialized accelerator.


In [43]:
import pandas as pd
import numpy as np
import os
import json
from tqdm import tqdm
from pydantic import BaseModel, Field
from typing import Literal
from openai import OpenAI
import random
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import safetensors
import umap
import hdbscan
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go
import textwrap
import gc
import re
import requests
import html
import re
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [44]:
pd.set_option('display.max_columns', None)

# Speech Embeddings

We load back the annotated data, keep only the SUBSTANTIVE_ARGUMENT paragraphs, remove missing bill title entries and then encode the paragraphs.

We save safetensors along with metadata to recover the data mapping.

In [45]:
df_speech = pd.read_csv("filtered_speech_level_data.csv")
df_speech.shape

(3141, 35)

In [46]:
df_speech

,Speech_ID,speaker_uid,speaker_first_name,speaker_last_name,speaker_fullname,party,party_name,gender,source_name,source_abbrev,meeting_uid,meeting_number,meeting_datetime,meeting_date,session_name,year,legislative_period_uid,Debate_ID,Bill_Number,Bill_Title_DE,Bill_Title_FR,Bill_Labels,Discussion_ID,Partition_ID,Paragraph_ID,name,role,language_of_text,TEXT_SPAN,lang,Affair_ID,title,keywords,ideological_summary,statutory_text
0,83161,1149.0,Urs,Schwaller,Schwaller Urs,CVP,Christlichdemokratische Volkspartei der Schweiz,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Schwaller Urs,For the committee,DE,Ein einziger Satz: Wir waren hier der Auffassu...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
1,83162,425.0,Rolf,Schweiger,Schweiger Rolf,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Schweiger Rolf,NaN,DE,Ich begründe meinen Antrag ganz kurz: Gesetzli...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
2,83163,276.0,Christoffel,Brändli,Brändli Christoffel,SVP,Schweizerische Volkspartei,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Brändli Christoffel,President,DE,Die Kommission beantragt aufgrund der Abstimmu...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
3,83164,1149.0,Urs,Schwaller,Schwaller Urs,CVP,Christlichdemokratische Volkspartei der Schweiz,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Schwaller Urs,For the committee,DE,Ich beantrage Ihnen hier die Streichung von Ab...,DE,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
4,83166,50.0,Pascal,Couchepin,Couchepin Pascal,FDP-Liberale,FDP.Die Liberalen,m,Online DB,ODB,962.0,2.0,2008-03-04T08:05:00.000000000,2008-03-04,Spring session,2008.0,48.0,04_476.4,4.476,Schutz der Bevölkerung und der Wirtschaft vor ...,Protection de la population et de l'économie c...,Bill|Parliamentary Initiative,13158.0,13158.1,1,Couchepin Pascal,President of the federal council,FR,"Si vous biffez cette disposition, vous risquez...",FR,20040476,Protection de la population et de l'économie c...,"Tabakkonsum, Nikotinsucht, Gesundheitszustand,...",Situation de départ et action nécessaire Chaqu...,"Conformément à l'art. 160, al. 1, de la Consti..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3136,324986,3999.0,Katharina,Prelicz-Huber,Prelicz-Huber Katharina,Grüne,Grüne Partei,f,Online DB,ODB,4803.0,4.0,2023-09-13T15:00:00.000000000,2023-09-13,Fall session,2023.0,51.0,09_528.5,9.528,Finanzieru

In [47]:
gc.collect()

1626

In [34]:
model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"attn_implementation": "flash_attention_2", "device_map": "auto"},
    processor_kwargs={"padding_side": "left"}
)
model

Loading weights: 100% 310/310 [00:32<00:00,  9.40it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Qwen3Model'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'lasttoken', 'include_prompt': True})
  (2): Normalize({})
)

In [49]:
speech_embeddings = model.encode(
    df_speech['TEXT_SPAN'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)
speech_embeddings.shape

Batches:   0% 0/13 [00:00<?, ?it/s]

Batches: 100% 13/13 [00:26<00:00,  2.00s/it]


(3141, 1024)

In [50]:
speech_ids_tensor = torch.tensor(df_speech['Speech_ID'].values)
speech_ids_tensor.shape

torch.Size([3141])

In [51]:
speech_embeddings_tensor = torch.from_numpy(speech_embeddings)
speech_embeddings_tensor.shape

torch.Size([3141, 1024])

In [52]:
safetensors.torch.save_file(
    tensors={
        "SpeechID": speech_ids_tensor,
        "speech_embeddings": speech_embeddings_tensor,
    },
    filename="speech_embeddings.safetensors",
    metadata={
        "SpeechID": "Unique identifier for each speech",
        "speech_embeddings": "Normalized embeddings of the speech paragraphs"
    }
)

In [53]:
from safetensors import safe_open

tensors = {}
with safe_open("speech_embeddings.safetensors", framework="pt", device=0) as f:
    for k in f.keys():
        tensors[k] = f.get_tensor(k)

speech_embeddings = tensors["speech_embeddings"].cpu().numpy()

# Unified Pre-Encoding

Instead of encoding two separate embeddings for the speech and the bill data, we use a single unified document string for each speech before passing it to the embedding model.


In [69]:
def format_speech_document(row, metadata_obj: dict) -> str:
    title = metadata_obj["title"].strip()
    keywords = metadata_obj["keywords"].strip()
    #ideological_summary = metadata_obj["ideological_summary"].strip()
    #statutory_text = metadata_obj["statutory_text"].strip()
    speech_text = str(row["TEXT_SPAN"]).strip()

    context_elements = []
    if title:
        context_elements.append(f"Bill Title: {title}")
    if keywords:
        context_elements.append(f"Keywords: {keywords}")
    #if ideological_summary:
    #    context_elements.append(f"Ideological Summary: {ideological_summary}")
    #if statutory_text:
    #    context_elements.append(f"Statutory Text: {statutory_text}")

    context_str = "\n".join(context_elements)

    formatted_document = f"{context_str}\n\nSpeech:\n{speech_text}"
    return formatted_document

In [70]:
speech_documents = df_speech.apply(
    lambda r: format_speech_document(r, {
        "title": str(r["title"]),
        "keywords": str(r["keywords"]),
        #"ideological_summary": str(r["ideological_summary"]),
        #"statutory_text": str(r["statutory_text"])
    }),
    axis=1
).tolist()

In [71]:
speech_documents

["Bill Title: Protection de la population et de l'économie contre le tabagisme passif\nKeywords: Tabakkonsum, Nikotinsucht, Gesundheitszustand, Gaststättengewerbe, Gastronomieberuf, öffentliches Gebäude, Arbeitsstätte, Schule, Spital, öffentlicher Verkehr\n\nSpeech:\nEin einziger Satz: Wir waren hier der Auffassung, dass der Bundesrat den tatsächlichen Gegebenheiten bei der Festsetzung des Datums des Inkrafttretens Rechnung tragen würde und auch den entsprechenden Zeitraum einräumen wird.",
 "Bill Title: Protection de la population et de l'économie contre le tabagisme passif\nKeywords: Tabakkonsum, Nikotinsucht, Gesundheitszustand, Gaststättengewerbe, Gastronomieberuf, öffentliches Gebäude, Arbeitsstätte, Schule, Spital, öffentlicher Verkehr\n\nSpeech:\nIch begründe meinen Antrag ganz kurz: Gesetzliche Änderungen bedingen immer Verhaltensweisen derjenigen, die von den entsprechenden Bestimmungen betroffen sind. Für die Gastbetriebe heisst das, dass sie allenfalls Konzepte umstellen, ba

In [72]:
task_prompt = "Represent this parliamentary speech for clustering by specific socio-economic and legal sub-issues:"

In [73]:
enriched_embeddings = model.encode(
    speech_documents,
    prompt=task_prompt,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100% 99/99 [00:30<00:00,  3.26it/s]


In [74]:
print("Enriched Embeddings Shape:", enriched_embeddings.shape)

Enriched Embeddings Shape: (3141, 1024)


In [75]:
safetensors.torch.save_file(
    tensors={
        "SpeechID": speech_ids_tensor,
        "enriched_embeddings": torch.from_numpy(enriched_embeddings),
    },
    filename="enriched_embeddings.safetensors",
    metadata={
        "SpeechID": "Unique identifier for each speech",
        "enriched_embeddings": "Enriched embeddings of the speech documents",
    }
)

# UMAP

We perform dimensionality reduction on the contextualized embeddings using UMAP to avoid having sparsity which might affect the performance of the HDBSCAN.

In [76]:
from safetensors import safe_open

tensors = {}
with safe_open("enriched_embeddings.safetensors", framework="pt", device=0) as f:
    for k in f.keys():
        tensors[k] = f.get_tensor(k)

In [77]:
enriched_embeddings = tensors['enriched_embeddings'].cpu().numpy()
enriched_embeddings.shape

(3141, 1024)

In [78]:
reducer_3d = umap.UMAP(n_components=3, metric='cosine', random_state=42)
embeddings_3d = reducer_3d.fit_transform(speech_embeddings)

reducer_3d_enriched = umap.UMAP(n_components=3, metric='cosine', random_state=42)
embeddings_3d_enriched = reducer_3d_enriched.fit_transform(enriched_embeddings)

build_algo set to brute_force_knn because random_state is given
build_algo set to brute_force_knn because random_state is given


In [79]:
import plotly.graph_objects as go

fig1 = go.Figure(
    data=[
        go.Scatter3d(
            x=embeddings_3d[:, 0],
            y=embeddings_3d[:, 1],
            z=embeddings_3d[:, 2],
            mode="markers",
            marker=dict(size=4, opacity=0.8),
        )
    ]
)

fig1.update_layout(
    title="3D Projection of Speech Embeddings",
    scene=dict(
        xaxis_title="UMAP Dimension 1",
        yaxis_title="UMAP Dimension 2",
        zaxis_title="UMAP Dimension 3",
    ),
    width=800,
    height=600,
)

fig1.show()


In [80]:
fig2 = go.Figure(
    data=[
        go.Scatter3d(
            x=embeddings_3d_enriched[:, 0],
            y=embeddings_3d_enriched[:, 1],
            z=embeddings_3d_enriched[:, 2],
            mode="markers",
            marker=dict(size=4, opacity=0.8),
        )
    ]
)

fig2.update_layout(
    title="3D Projection of Enriched Argument Embeddings",
    scene=dict(
        xaxis_title="UMAP Dimension 1",
        yaxis_title="UMAP Dimension 2",
        zaxis_title="UMAP Dimension 3",
    ),
    width=800,
    height=600,
)

fig2.show()
